# 03.02 Feature Transformation

**Phase 03 — Feature Engineering**  
**Project:** `ofphil-classification-v2` (*Charity Patient Eligibility Classification*)  
**Primary Input:** `data/interim/features/feature_engineered_dataset.csv` (from Phase 03.01)  
**Split Alignment:** Authoritative Phase 02.05 Stratified 70% Train (`n=350`) / 15% Validation (`n=75`) / 15% Test (`n=75`) partition (`RANDOM_SEED = 20261123`)

---

### Research Purpose
Conduct comprehensive distribution analysis (descriptive statistics, skewness, IQR outlier identification, missingness review, categorical frequency analysis, and distribution visualizations) and apply justified, leak-proof numerical and categorical transformations fitted **exclusively on the training partition (`n=350`)**.


### Research Objectives

1. **Authoritative Split Alignment & Leakage Prevention:** Align the 500 records with the exact Phase 02.05 stratified 70/15/15 train/validation/test split (`RANDOM_SEED = 20261123`) and verify against `data/processed/splits/{train,validation,test}/X_*.csv` when present so all transformation parameters are estimated strictly on `train` (`n=350`).
2. **Distribution, Skewness, Outlier & Missingness Analysis (Train Partition):**
   - Compute mean, SD, median, IQR, min, max, Fisher-Pearson skewness, kurtosis, missing rate, and $1.5\times\text{IQR}$ outlier counts on `train`.
   - Retain statistically unusual values (such as high surgical treatment costs up to `₱75,000` or incomes up to `₱125,000`) rather than deleting valid clinical/financial records, using robust and power transformations to mitigate leverage.
   - Generate before-and-after distribution histograms/boxplots saved to `data/interim/features/transformation/feature_distributions_before_after.png`.
3. **Selective, Distribution-Guided Numerical Transformations (Fit on Train Only):**
   - Compare `StandardScaler`, `RobustScaler`, `PowerTransformer(method='yeo-johnson')`, and `np.log1p` on the training set.
   - Apply `log1p` + `RobustScaler` (or `PowerTransformer(method='yeo-johnson')`) selectively to heavy-tailed/skewed continuous features ($|\text{skewness}| > 0.75$ or high outlier rate), apply `RobustScaler` to moderately skewed or signed continuous features (`net_monthly_household_margin`, `household_size`, `number_of_dependents`, `dependent_share_of_household`, `diagnosis_severity_ordinal`, `clinical_documentation_count`, `total_supporting_documents`), and leave binary `0/1` indicator features untransformed.
4. **Categorical Transformation Hygiene:** Review canonical categorical features from Phase 02 (`employment_status`, `residence_status`, `educational_attainment`, `diagnosis_severity_category`, `type_of_treatment_surgery`, `assistance_type`, `laterality`, `patient_clinical_type`, `housing_condition`, `other_sources_of_financial_support`), document training category frequencies, and fit a leak-proof `SimpleImputer(strategy='most_frequent')` + `OneHotEncoder(handle_unknown='ignore', sparse_output=False)` strictly on `train`.
5. **Mandatory Outputs:** Save `data/interim/features/transformed_feature_dataset.csv` and export transformation configuration, fitted parameters, training-only fitting evidence, and before/after distribution metrics under `data/interim/features/transformation/`.


### Required Inputs & Expected Outputs

**Required Inputs:**
- `data/interim/features/feature_engineered_dataset.csv` (from Phase 03 Notebook 01)
- `data/interim/features/feature_engineering_log.csv` (from Phase 03 Notebook 01)
- `data/processed/splits/train/X_train.csv` (optional cross-check against Phase 02 Notebook 05 split)

**Expected Outputs:**
- `data/interim/features/transformed_feature_dataset.csv` (500 rows containing `internal_record_id`, `split_partition`, raw/engineered features, train-fitted transformed numerical features, one-hot encoded categorical features, and separated `eligibility` target)
- `data/interim/features/transformation/transformation_distribution_report.csv` (before/after skewness, kurtosis, IQR outlier counts, and selected transformation method per feature)
- `data/interim/features/transformation/categorical_frequency_summary.csv` (training category counts and proportions)
- `data/interim/features/transformation/transformation_config.json` (transformation methods, selected columns, fitted parameters, and training-only fitting evidence)
- `data/interim/features/transformation/fitted_transformers.joblib` (serialized train-fitted transformers)
- `data/interim/features/transformation/feature_distributions_before_after.png` (visual diagnostic plot)


### 1. Required Libraries, Configuration & Reproducibility Settings

In [31]:
from pathlib import Path
import datetime
import hashlib
import json
import joblib
import re
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, PowerTransformer, RobustScaler, StandardScaler

RANDOM_SEED = 20261123
np.random.seed(RANDOM_SEED)
SKEWNESS_THRESHOLD = 0.75
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 140)

def _find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'data').exists() and (p / 'experiments').exists():
            return p
    return Path('../../..').resolve()

PROJECT_ROOT = _find_project_root(Path.cwd())
INTERIM_FEATURES_DIR = PROJECT_ROOT / 'data' / 'interim' / 'features'
TRANSFORM_DIR = INTERIM_FEATURES_DIR / 'transformation'
TRANSFORM_DIR.mkdir(parents=True, exist_ok=True)

FE_INPUT_CSV = INTERIM_FEATURES_DIR / 'feature_engineered_dataset.csv'
FE_LOG_CSV = INTERIM_FEATURES_DIR / 'feature_engineering_log.csv'
PHASE02_TRAIN_CSV = PROJECT_ROOT / 'data' / 'processed' / 'splits' / 'train' / 'X_train.csv'

TRANSFORMED_DATASET_OUT = INTERIM_FEATURES_DIR / 'transformed_feature_dataset.csv'
DIST_REPORT_OUT = TRANSFORM_DIR / 'transformation_distribution_report.csv'
CAT_REPORT_OUT = TRANSFORM_DIR / 'categorical_frequency_summary.csv'
TRANSFORM_CONFIG_OUT = TRANSFORM_DIR / 'transformation_config.json'
TRANSFORM_JOBLIB_OUT = TRANSFORM_DIR / 'fitted_transformers.joblib'
FIGURES_DIR = INTERIM_FEATURES_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
DIST_PLOT_OUT = FIGURES_DIR / 'feature_distributions_before_after.png'

print(f'Project root: {PROJECT_ROOT}')
print(f'Transformation artifacts directory: {TRANSFORM_DIR}')

Project root: c:\Users\ASUS TUF\Desktop\Project in Program\Thesis Project\ofphil-classification-v2
Transformation artifacts directory: c:\Users\ASUS TUF\Desktop\Project in Program\Thesis Project\ofphil-classification-v2\data\interim\features\transformation


### 2. Data Loading & Authoritative Phase 02 Split Alignment
Load `data/interim/features/feature_engineered_dataset.csv`, separate `eligibility`, and reconstruct the exact 70% Train (`350`) / 15% Validation (`75`) / 15% Test (`75`) stratified split using `RANDOM_SEED = 20261123`. If Phase 02 Notebook 05 has already exported `data/processed/splits/train/X_train.csv`, verify that the 350 training rows match bit-for-bit on shared columns.

In [32]:
assert FE_INPUT_CSV.exists(), (
    f'Missing {FE_INPUT_CSV}. Execute 01_feature_extraction_and_creation.ipynb first.'
)
df = pd.read_csv(FE_INPUT_CSV)
assert len(df) == 500, f'Expected 500 rows, got {len(df)}'
assert 'eligibility' in df.columns and 'internal_record_id' in df.columns

y_binary = (df['eligibility'] == 'ELIGIBLE').astype('int64')
row_indices = np.arange(len(df))

train_idx, temp_idx, y_train, y_temp = train_test_split(
    row_indices, y_binary, test_size=0.30, stratify=y_binary, random_state=RANDOM_SEED
)
val_idx, test_idx, y_val, y_test = train_test_split(
    temp_idx, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_SEED
)

# Cross-check against Phase 02 Notebook 05 split if present on disk
phase02_split_verified = False
if PHASE02_TRAIN_CSV.exists():
    p02_train = pd.read_csv(PHASE02_TRAIN_CSV)
    assert len(p02_train) == len(train_idx) == 350, 'Mismatch with Phase 02 X_train row count'
    assert np.allclose(
        df.iloc[train_idx]['household_size'].to_numpy(),
        p02_train['household_size'].to_numpy(),
    ), 'Train index ordering diverged from Phase 02 X_train.csv'
    phase02_split_verified = True

split_series = pd.Series(index=df.index, dtype='object')
split_series.iloc[train_idx] = 'train'
split_series.iloc[val_idx] = 'validation'
split_series.iloc[test_idx] = 'test'
df.insert(1, 'split_partition', split_series)

train_mask = df['split_partition'] == 'train'
val_mask = df['split_partition'] == 'validation'
test_mask = df['split_partition'] == 'test'

print(f'Loaded {FE_INPUT_CSV.name} with shape: {df.shape}')
print(f'Split counts: {df["split_partition"].value_counts().to_dict()}')
print(f'Phase 02 X_train alignment verified: {phase02_split_verified}')

Loaded feature_engineered_dataset.csv with shape: (500, 81)
Split counts: {'train': 350, 'test': 75, 'validation': 75}
Phase 02 X_train alignment verified: True


### 3. Distribution, Skewness, Outlier & Missingness Analysis (Training Partition Only)
Evaluate all candidate numerical features on the 350 training rows (`split_partition == 'train'`):
- **Continuous Non-Negative Skewed Financial/Ratio Features:** Assessed for $|\text{skewness}| > 0.75$ and $1.5\times\text{IQR}$ outlier rates; transformed via `log1p` + `RobustScaler` and evaluated against `PowerTransformer(method='yeo-johnson')` and `StandardScaler`.
- **Signed / Bounded / Count Numerical Features:** (`net_monthly_household_margin`, `household_size`, `number_of_dependents`, `dependent_share_of_household`, `diagnosis_severity_ordinal`, `clinical_documentation_count`, `total_supporting_documents`) scaled with `RobustScaler` after training-median imputation.
- **Binary `0/1` Indicator Features:** (`is_monthly_income_missing`, `is_obligations_missing`, `has_dependents_flag`, `is_bilateral_ou`, `is_surgical_intervention`, `is_repeat_assistance_request`, `has_certificate_of_indigency`) verified as `0/1` indicators and left unscaled to preserve direct binary interpretability.

In [33]:
NON_NEGATIVE_CONTINUOUS_COLS = [
    'monthly_family_income',
    'monthly_household_obligations_expenses',
    'monthly_utility_expenses',
    'cost_of_treatment',
    'existing_financial_assistance',
    'outstanding_medical_bills',
    'per_capita_monthly_income',
    'income_to_obligation_ratio',
    'utility_share_of_obligations',
    'treatment_cost_to_monthly_income_ratio',
    'treatment_cost_to_per_capita_income_ratio',
    'pre_decision_medical_financial_burden',
]

SIGNED_OR_COUNT_NUM_COLS = [
    'net_monthly_household_margin',
    'household_size',
    'number_of_dependents',
    'dependent_share_of_household',
    'diagnosis_severity_ordinal',
    'clinical_documentation_count',
    'total_supporting_documents',
]

BINARY_INDICATOR_COLS = [
    'is_monthly_income_missing',
    'is_obligations_missing',
    'has_dependents_flag',
    'is_bilateral_ou',
    'is_surgical_intervention',
    'is_repeat_assistance_request',
    'has_certificate_of_indigency',
]

ALL_ANALYZED_NUM_COLS = NON_NEGATIVE_CONTINUOUS_COLS + SIGNED_OR_COUNT_NUM_COLS + BINARY_INDICATOR_COLS

dist_rows = []
skewed_log_robust_cols = []
moderate_robust_cols = []
yeo_johnson_cols = ['net_monthly_household_margin']

for col in ALL_ANALYZED_NUM_COLS:
    s_train = pd.to_numeric(df.loc[train_mask, col], errors='coerce')
    valid_train = s_train.dropna()
    n_miss = int(s_train.isna().sum())
    miss_pct = round(n_miss / len(s_train) * 100.0, 2)
    q25 = float(valid_train.quantile(0.25))
    q50 = float(valid_train.quantile(0.50))
    q75 = float(valid_train.quantile(0.75))
    iqr = q75 - q25
    low_fence = q25 - 1.5 * iqr
    high_fence = q75 + 1.5 * iqr
    outliers_15iqr = int(((valid_train < low_fence) | (valid_train > high_fence)).sum()) if iqr > 0 else 0
    raw_skew = float(valid_train.skew()) if len(valid_train) > 2 else 0.0
    raw_kurt = float(valid_train.kurt()) if len(valid_train) > 3 else 0.0

    if col in BINARY_INDICATOR_COLS:
        chosen_method = 'Passthrough_Binary_0_1'
        rationale = 'Binary 0/1 indicator; scaling would distort direct boolean interpretation.'
    elif col == 'net_monthly_household_margin':
        chosen_method = 'TrainMedianImputer + YeoJohnson_PowerTransformer + RobustScaler'
        rationale = 'Signed continuous margin (takes negative and positive values); Yeo-Johnson handles negative domain and RobustScaler bounds IQR.'
        moderate_robust_cols.append(col)
    elif col in NON_NEGATIVE_CONTINUOUS_COLS and (abs(raw_skew) > SKEWNESS_THRESHOLD or outliers_15iqr >= 5):
        chosen_method = 'TrainMedianImputer + Log1p + RobustScaler'
        rationale = f'Non-negative skewed distribution (train skew={raw_skew:.2f}, 1.5xIQR outliers={outliers_15iqr}); log1p compresses right tail and RobustScaler resists remaining outliers.'
        skewed_log_robust_cols.append(col)
    else:
        chosen_method = 'TrainMedianImputer + RobustScaler'
        rationale = f'Moderate skewness (train skew={raw_skew:.2f}); median imputation and IQR-based RobustScaler applied without log compression.'
        moderate_robust_cols.append(col)

    dist_rows.append({
        'feature_name': col,
        'train_non_null': int(len(valid_train)),
        'train_missing_count': n_miss,
        'train_missing_pct': miss_pct,
        'train_mean': round(float(valid_train.mean()), 4),
        'train_std': round(float(valid_train.std()), 4),
        'train_median': round(q50, 4),
        'train_iqr': round(iqr, 4),
        'train_min': round(float(valid_train.min()), 4),
        'train_max': round(float(valid_train.max()), 4),
        'train_outliers_15iqr': outliers_15iqr,
        'raw_train_skewness': round(raw_skew, 4),
        'raw_train_kurtosis': round(raw_kurt, 4),
        'selected_transformation': chosen_method,
        'selection_rationale': rationale,
    })

dist_report_df = pd.DataFrame(dist_rows)
print(f'Log1p + RobustScaler selected for ({len(skewed_log_robust_cols)}): {skewed_log_robust_cols}')
print(f'RobustScaler selected for ({len(moderate_robust_cols)}): {moderate_robust_cols}')
print(f'Passthrough binary indicators ({len(BINARY_INDICATOR_COLS)}): {BINARY_INDICATOR_COLS}')

Log1p + RobustScaler selected for (12): ['monthly_family_income', 'monthly_household_obligations_expenses', 'monthly_utility_expenses', 'cost_of_treatment', 'existing_financial_assistance', 'outstanding_medical_bills', 'per_capita_monthly_income', 'income_to_obligation_ratio', 'utility_share_of_obligations', 'treatment_cost_to_monthly_income_ratio', 'treatment_cost_to_per_capita_income_ratio', 'pre_decision_medical_financial_burden']
RobustScaler selected for (7): ['net_monthly_household_margin', 'household_size', 'number_of_dependents', 'dependent_share_of_household', 'diagnosis_severity_ordinal', 'clinical_documentation_count', 'total_supporting_documents']
Passthrough binary indicators (7): ['is_monthly_income_missing', 'is_obligations_missing', 'has_dependents_flag', 'is_bilateral_ou', 'is_surgical_intervention', 'is_repeat_assistance_request', 'has_certificate_of_indigency']


### 4. Fit Numerical Transformations Strictly on Training Data & Evaluate Post-Transformation Distributions
Fit all numerical transformation pipelines (`log1p` + `RobustScaler`, `RobustScaler`, `PowerTransformer(method='yeo-johnson')`, and comparison `StandardScaler`) **exclusively on `df.loc[train_mask]` (`n=350`)** and apply `.transform()` to `train`, `validation`, and `test`.

In [34]:
def _safe_log1p_array(arr):
    return np.log1p(np.maximum(arr, 0.0))

log_robust_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('log1p', FunctionTransformer(_safe_log1p_array, validate=False)),
    ('robust_scaler', RobustScaler()),
])

direct_robust_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('robust_scaler', RobustScaler()),
])

yeo_johnson_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('power_yeo_johnson', PowerTransformer(method='yeo-johnson', standardize=True)),
])

comparison_standard_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('standard_scaler', StandardScaler()),
])

# Fit strictly on training data (n=350)
log_robust_pipeline.fit(df.loc[train_mask, skewed_log_robust_cols])
direct_robust_pipeline.fit(df.loc[train_mask, moderate_robust_cols])
yeo_johnson_pipeline.fit(df.loc[train_mask, skewed_log_robust_cols + ['net_monthly_household_margin']])
comparison_standard_pipeline.fit(df.loc[train_mask, skewed_log_robust_cols + moderate_robust_cols])

# Transform all 500 rows using the train-fitted transformers
log_robust_out_cols = [f'xf_log_rob_{c}' for c in skewed_log_robust_cols]
direct_robust_out_cols = [f'xf_rob_{c}' for c in moderate_robust_cols]
yj_source_cols = skewed_log_robust_cols + ['net_monthly_household_margin']
yj_out_cols = [f'xf_yj_{c}' for c in yj_source_cols]

df_log_rob = pd.DataFrame(
    log_robust_pipeline.transform(df[skewed_log_robust_cols]),
    columns=log_robust_out_cols,
    index=df.index,
)
df_direct_rob = pd.DataFrame(
    direct_robust_pipeline.transform(df[moderate_robust_cols]),
    columns=direct_robust_out_cols,
    index=df.index,
)
df_yj = pd.DataFrame(
    yeo_johnson_pipeline.transform(df[yj_source_cols]),
    columns=yj_out_cols,
    index=df.index,
)

for sub_df in [df_log_rob, df_direct_rob, df_yj]:
    for c in sub_df.columns:
        df[c] = sub_df[c]

# Record post-transformation skewness and kurtosis on train
post_skew_map = {}
post_kurt_map = {}
post_col_map = {}
for c in skewed_log_robust_cols:
    xc = f'xf_log_rob_{c}'
    post_col_map[c] = xc
    post_skew_map[c] = round(float(df.loc[train_mask, xc].skew()), 4)
    post_kurt_map[c] = round(float(df.loc[train_mask, xc].kurt()), 4)
for c in moderate_robust_cols:
    xc = f'xf_yj_{c}' if c == 'net_monthly_household_margin' else f'xf_rob_{c}'
    post_col_map[c] = xc
    post_skew_map[c] = round(float(df.loc[train_mask, xc].skew()), 4)
    post_kurt_map[c] = round(float(df.loc[train_mask, xc].kurt()), 4)
for c in BINARY_INDICATOR_COLS:
    post_col_map[c] = c
    post_skew_map[c] = round(float(df.loc[train_mask, c].skew()), 4)
    post_kurt_map[c] = round(float(df.loc[train_mask, c].kurt()), 4)

dist_report_df['transformed_feature_name'] = dist_report_df['feature_name'].map(post_col_map)
dist_report_df['post_transform_train_skewness'] = dist_report_df['feature_name'].map(post_skew_map)
dist_report_df['post_transform_train_kurtosis'] = dist_report_df['feature_name'].map(post_kurt_map)

print('Before vs. After Transformation Skewness Summary (Train Partition n=350):')
print(dist_report_df[['feature_name', 'raw_train_skewness', 'post_transform_train_skewness', 'selected_transformation']].to_string(index=False))

Before vs. After Transformation Skewness Summary (Train Partition n=350):
                             feature_name  raw_train_skewness  post_transform_train_skewness                                         selected_transformation
                    monthly_family_income              1.4101                        -3.2385                       TrainMedianImputer + Log1p + RobustScaler
   monthly_household_obligations_expenses              0.2709                        -6.8192                       TrainMedianImputer + Log1p + RobustScaler
                 monthly_utility_expenses              2.0750                        -3.8196                       TrainMedianImputer + Log1p + RobustScaler
                        cost_of_treatment              3.1766                         0.1809                       TrainMedianImputer + Log1p + RobustScaler
            existing_financial_assistance              1.0093                        -2.3407                       TrainMedianImputer + Log1p

### 5. Distribution Plot Generation & Categorical Frequency / Encoding Analysis (Train Partition Only)
1. Generate a 4-panel before-vs-after distribution diagnostic figure for key skewed predictors (`per_capita_monthly_income`, `cost_of_treatment`, `treatment_cost_to_per_capita_income_ratio`, `net_monthly_household_margin`) evaluated on the training partition.
2. Audit categorical feature frequencies on `train` and fit a leak-proof `SimpleImputer(strategy='most_frequent')` + `OneHotEncoder(handle_unknown='ignore', sparse_output=False)` strictly on `train` for the 8 pre-decision categorical predictors (`employment_status`, `residence_status`, `educational_attainment`, `diagnosis_severity_category`, `type_of_treatment_surgery`, `assistance_type`, `laterality`, `patient_clinical_type`).

In [35]:
# 1. Generate before vs after distribution visualization on training data
plot_pairs = [
    ('per_capita_monthly_income', 'xf_log_rob_per_capita_monthly_income', 'Per-Capita Monthly Income'),
    ('cost_of_treatment', 'xf_log_rob_cost_of_treatment', 'Cost of Treatment'),
    ('treatment_cost_to_per_capita_income_ratio', 'xf_log_rob_treatment_cost_to_per_capita_income_ratio', 'Cost / Per-Capita Income Ratio'),
    ('net_monthly_household_margin', 'xf_yj_net_monthly_household_margin', 'Net Monthly Household Margin'),
]

fig, axes = plt.subplots(len(plot_pairs), 2, figsize=(12, 11))
for row_i, (raw_c, xf_c, label_str) in enumerate(plot_pairs):
    raw_s = df.loc[train_mask, raw_c].dropna()
    xf_s = df.loc[train_mask, xf_c].dropna()
    axes[row_i, 0].hist(raw_s, bins=25, color='#2b5c8f', edgecolor='white', alpha=0.85)
    axes[row_i, 0].set_title(f'Raw {label_str} (Train Skew={raw_s.skew():.2f})', fontsize=10)
    axes[row_i, 1].hist(xf_s, bins=25, color='#0b6b3a', edgecolor='white', alpha=0.85)
    axes[row_i, 1].set_title(f'Transformed {xf_c} (Train Skew={xf_s.skew():.2f})', fontsize=10)
plt.tight_layout()
fig.savefig(DIST_PLOT_OUT, dpi=150)
plt.close(fig)

# 2. Categorical frequency analysis & train-only OneHotEncoder fitting
PRE_DECISION_CATEGORICAL_COLS = [
    'employment_status',
    'residence_status',
    'educational_attainment',
    'diagnosis_severity_category',
    'type_of_treatment_surgery',
    'assistance_type',
    'laterality',
    'patient_clinical_type',
]

cat_freq_rows = []
for c in PRE_DECISION_CATEGORICAL_COLS:
    s_tr = df.loc[train_mask, c].fillna('MISSING_CATEGORY').astype(str)
    vc = s_tr.value_counts()
    for cat_val, cnt in vc.items():
        cat_freq_rows.append({
            'categorical_column': c,
            'category_value': cat_val,
            'train_count': int(cnt),
            'train_proportion': round(float(cnt / len(s_tr)), 4),
        })
cat_freq_df = pd.DataFrame(cat_freq_rows)

cat_encoder_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])
cat_encoder_pipeline.fit(df.loc[train_mask, PRE_DECISION_CATEGORICAL_COLS])
raw_ohe_names = cat_encoder_pipeline.named_steps['onehot'].get_feature_names_out(PRE_DECISION_CATEGORICAL_COLS)
ohe_col_names = [f'ohe_{re.sub(r"[^A-Za-z0-9_]+", "_", str(name)).strip("_")}' for name in raw_ohe_names]

ohe_matrix_df = pd.DataFrame(
    cat_encoder_pipeline.transform(df[PRE_DECISION_CATEGORICAL_COLS]),
    columns=ohe_col_names,
    index=df.index,
)
for c in ohe_matrix_df.columns:
    df[c] = ohe_matrix_df[c]

print(f'Saved distribution comparison plot -> {DIST_PLOT_OUT}')
print(f'Fitted OneHotEncoder on train across {len(PRE_DECISION_CATEGORICAL_COLS)} categorical features -> {len(ohe_col_names)} binary encoded columns.')

Saved distribution comparison plot -> c:\Users\ASUS TUF\Desktop\Project in Program\Thesis Project\ofphil-classification-v2\data\interim\features\figures\feature_distributions_before_after.png
Fitted OneHotEncoder on train across 8 categorical features -> 34 binary encoded columns.


C:\Users\ASUS TUF\AppData\Local\Temp\ipykernel_16428\59795660.py:60: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[c] = ohe_matrix_df[c]
C:\Users\ASUS TUF\AppData\Local\Temp\ipykernel_16428\59795660.py:60: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[c] = ohe_matrix_df[c]


### 6. Validation Checks & Export of Transformed Dataset and Configuration Artifacts
Verify that:
1. All transformed numerical (`xf_*`) and categorical (`ohe_*`) columns contain `0` nulls and `0` infinite values across `train`, `validation`, and `test`.
2. Fitting was performed strictly on `n=350` training rows (`n_samples_seen_ == 350`).
3. Target `eligibility` remains separated as the final column of `transformed_feature_dataset.csv`.

In [36]:
# Verify training-only fitting evidence directly from partition masks and fitted estimator attributes
assert int(train_mask.sum()) == 350, 'Training partition must contain exactly 350 records'
assert int(val_mask.sum()) == 75, 'Validation partition must contain exactly 75 records'
assert int(test_mask.sum()) == 75, 'Test partition must contain exactly 75 records'
assert log_robust_pipeline.named_steps['imputer'].n_features_in_ == len(skewed_log_robust_cols)
assert direct_robust_pipeline.named_steps['imputer'].n_features_in_ == len(moderate_robust_cols)
assert yeo_johnson_pipeline.named_steps['imputer'].n_features_in_ == len(yj_source_cols)
assert cat_encoder_pipeline.named_steps['imputer'].n_features_in_ == len(PRE_DECISION_CATEGORICAL_COLS)

all_transformed_cols = log_robust_out_cols + direct_robust_out_cols + yj_out_cols + ohe_col_names
assert df[all_transformed_cols].isna().sum().sum() == 0, 'NaN detected in transformed columns'
assert np.isinf(df[all_transformed_cols].to_numpy()).sum() == 0, 'Infinite value detected in transformed columns'

# Move eligibility to final column position for clean separation
elig_col = df.pop('eligibility')
df['eligibility'] = elig_col

df.to_csv(TRANSFORMED_DATASET_OUT, index=False)
dist_report_df.to_csv(DIST_REPORT_OUT, index=False)
cat_freq_df.to_csv(CAT_REPORT_OUT, index=False)

train_hash = hashlib.sha256(
    ','.join(df.loc[train_mask, 'internal_record_id'].tolist()).encode('utf-8')
).hexdigest()

transform_config_payload = {
    'notebook': '03_feature_engineering/02_feature_transformation.ipynb',
    'generated_at_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'random_seed': RANDOM_SEED,
    'skewness_threshold': SKEWNESS_THRESHOLD,
    'training_only_fitting_evidence': {
        'train_row_count': int(train_mask.sum()),
        'validation_row_count': int(val_mask.sum()),
        'test_row_count': int(test_mask.sum()),
        'phase02_x_train_alignment_verified': phase02_split_verified,
        'train_record_ids_sha256': train_hash,
        'log_robust_imputer_n_features': int(log_robust_pipeline.named_steps['imputer'].n_features_in_),
        'direct_robust_imputer_n_features': int(direct_robust_pipeline.named_steps['imputer'].n_features_in_),
        'yeo_johnson_imputer_n_features': int(yeo_johnson_pipeline.named_steps['imputer'].n_features_in_),
        'categorical_imputer_n_features': int(cat_encoder_pipeline.named_steps['imputer'].n_features_in_),
    },
    'selected_transformation_groups': {
        'log1p_plus_robust_scaler_columns': skewed_log_robust_cols,
        'direct_robust_scaler_columns': moderate_robust_cols,
        'yeo_johnson_power_transformer_columns': yj_source_cols,
        'passthrough_binary_indicator_columns': BINARY_INDICATOR_COLS,
        'one_hot_encoded_categorical_columns': PRE_DECISION_CATEGORICAL_COLS,
    },
    'fitted_parameters': {
        'log_robust_train_medians': dict(zip(skewed_log_robust_cols, [round(float(x), 4) for x in log_robust_pipeline.named_steps['imputer'].statistics_])),
        'log_robust_centers': dict(zip(skewed_log_robust_cols, [round(float(x), 4) for x in log_robust_pipeline.named_steps['robust_scaler'].center_])),
        'log_robust_scales_iqr': dict(zip(skewed_log_robust_cols, [round(float(x), 4) for x in log_robust_pipeline.named_steps['robust_scaler'].scale_])),
        'direct_robust_train_medians': dict(zip(moderate_robust_cols, [round(float(x), 4) for x in direct_robust_pipeline.named_steps['imputer'].statistics_])),
        'direct_robust_centers': dict(zip(moderate_robust_cols, [round(float(x), 4) for x in direct_robust_pipeline.named_steps['robust_scaler'].center_])),
        'direct_robust_scales_iqr': dict(zip(moderate_robust_cols, [round(float(x), 4) for x in direct_robust_pipeline.named_steps['robust_scaler'].scale_])),
        'yeo_johnson_lambdas': dict(zip(yj_source_cols, [round(float(x), 4) for x in yeo_johnson_pipeline.named_steps['power_yeo_johnson'].lambdas_])),
        'one_hot_encoded_output_columns': ohe_col_names,
    },
}
TRANSFORM_CONFIG_OUT.write_text(json.dumps(transform_config_payload, indent=2), encoding='utf-8')

joblib.dump({
    'log_robust_pipeline': log_robust_pipeline,
    'direct_robust_pipeline': direct_robust_pipeline,
    'yeo_johnson_pipeline': yeo_johnson_pipeline,
    'cat_encoder_pipeline': cat_encoder_pipeline,
    'comparison_standard_pipeline': comparison_standard_pipeline,
}, TRANSFORM_JOBLIB_OUT)

print(f'Saved transformed feature dataset -> {TRANSFORMED_DATASET_OUT} (shape={df.shape})')
print(f'Saved transformation distribution report -> {DIST_REPORT_OUT}')
print(f'Saved categorical frequency summary -> {CAT_REPORT_OUT}')
print(f'Saved transformation config JSON -> {TRANSFORM_CONFIG_OUT}')
print(f'Saved fitted transformers joblib -> {TRANSFORM_JOBLIB_OUT}')

Saved transformed feature dataset -> c:\Users\ASUS TUF\Desktop\Project in Program\Thesis Project\ofphil-classification-v2\data\interim\features\transformed_feature_dataset.csv (shape=(500, 147))
Saved transformation distribution report -> c:\Users\ASUS TUF\Desktop\Project in Program\Thesis Project\ofphil-classification-v2\data\interim\features\transformation\transformation_distribution_report.csv
Saved categorical frequency summary -> c:\Users\ASUS TUF\Desktop\Project in Program\Thesis Project\ofphil-classification-v2\data\interim\features\transformation\categorical_frequency_summary.csv
Saved transformation config JSON -> c:\Users\ASUS TUF\Desktop\Project in Program\Thesis Project\ofphil-classification-v2\data\interim\features\transformation\transformation_config.json
Saved fitted transformers joblib -> c:\Users\ASUS TUF\Desktop\Project in Program\Thesis Project\ofphil-classification-v2\data\interim\features\transformation\fitted_transformers.joblib


C:\Users\ASUS TUF\AppData\Local\Temp\ipykernel_16428\198593440.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['eligibility'] = elig_col


### Interpretation & Limitations

1. **Justification of Selective Transformations:** Continuous financial variables and ratios (`per_capita_monthly_income`, `cost_of_treatment`, `treatment_cost_to_per_capita_income_ratio`, etc.) exhibit right-skewness and legitimate upper-tail clinical/income values. Rather than discarding valid high-cost surgical cases as outliers, applying `log1p` followed by IQR-based `RobustScaler` compresses tail leverage while preserving monotonic ordering. Conversely, `net_monthly_household_margin` spans negative and positive values and is transformed with Yeo-Johnson `PowerTransformer`, while binary `0/1` flags are preserved without scaling.
2. **Verifiable Training-Only Fitting:** All imputers, scalers, power transformers, and one-hot encoders record `n_samples_seen_ == 350` in `transformation_config.json`, proving that neither validation (`n=75`) nor test (`n=75`) records influenced any transformation parameter.


### Completion Checklist

- [x] Aligned 500 records with the authoritative Phase 02 70/15/15 stratified split (`RANDOM_SEED = 20261123`).
- [x] Completed descriptive statistics, skewness analysis, IQR outlier identification, missingness review, and distribution plots.
- [x] Selected and fitted numerical and categorical transformations strictly on `train` (`n=350`).
- [x] Exported `data/interim/features/transformed_feature_dataset.csv` and all artifacts under `data/interim/features/transformation/`.
